# Расчет гранов после намыва

### 1. Загрузка рабочего журнала по намыву

- Загружается журнал в заголовке несколько строк они соединяются и создается обычный заголовок с соединенными строками
- Столбцы переименовываются в рабочие


### 2. Загрузка тарировок

- Загрузка тарировок из эксель файла

### 3. Обработка и расчет

- У каждой пробы по две строки с помощью аггрегирующей функции создается таблица с данными в одну строку для каждой пробы для удобства работы
- Непосредственно сам расчет

In [17]:
import pandas as pd

In [18]:
def zagr_file(path):
    df = pd.read_excel(path, skiprows=4, header=[0, 1, 2], na_values="-")

    df.columns = [process_multiheader_column(col) for col in df.columns]

    df = df.rename(columns=COlUMNS)
    df = df[COlUMNS.values()]

    return df

def process_multiheader_column(col):
    if isinstance(col, tuple):
        clean_parts = []
        for part in col:
            if pd.isna(part):
                continue
            part_str = str(part)
            if part_str.startswith("Unnamed:"):
                continue
            if part_str.strip() and part_str != "nan":
                clean_parts.append(part_str.strip())
        return "_".join(clean_parts) if clean_parts else f"column_{hash(col) % 1000}"
    return str(col).strip()

def obrabotka_df_posle_zagr(df):

    df['lab_nomer'] = df['lab_nomer'].ffill()

    df_agg = df.groupby('lab_nomer').agg({**agg_dict, **temp_agg})

    df_agg.columns = [process_multiheader_column(col) for col in df_agg.columns]
    df_agg = df_agg.reset_index()

    df_agg[COLS_GRAN] = df_agg[COLS_GRAN].fillna(0)

    return df_agg

def zagr_tarirovki(path):
    df_tarirovk = pd.read_excel(path)
    df_tarirovk = df_tarirovk.set_index('№')

    return df_tarirovk

def rashet_popravki_areometr(df_agg, df_tarirovk):

    df_agg['popravka_t1'] = df_agg.apply(get_value1, axis=1, ref_df=df_tarirovk, col_temp='1_zamer/temp_last')
    df_agg['popravka_t2'] = df_agg.apply(get_value1, axis=1, ref_df=df_tarirovk, col_temp='2_zamer/temp_last')
    df_agg['popravka_t3'] = df_agg.apply(get_value1, axis=1, ref_df=df_tarirovk, col_temp='3_zamer/temp_last')

    df_agg['zamer_1'] = df_agg['1_zamer/temp_first'] + df_agg['popravka_t1']
    df_agg['zamer_2'] = df_agg['2_zamer/temp_first'] + df_agg['popravka_t2']
    df_agg['zamer_3'] = df_agg['3_zamer/temp_first'] + df_agg['popravka_t3']

    return df_agg

    return df_agg

def get_value1(row, ref_df, col_temp):
    idx = row['areometr_first']
    col = row[col_temp]
    # .at работает быстро для одиночного доступа по метке
    return ref_df.at[idx, col] if idx in ref_df.index and col in ref_df.columns else None

def rashet_x1_x2_x3(df_agg, udelka):
    df_agg['udelka'] = udelka

    df_agg['koef_K'] = (df_agg['gran_10_first'] + df_agg['gran_5-10_first'] + df_agg['gran_5-2_first'] + df_agg['gran_2-1_first'])

    df_agg = df_agg.dropna(subset=['3_zamer/temp_first'])

    df_agg['X1'] = df_agg['udelka'] * df_agg['zamer_1']/(df_agg['udelka']-1)/df_agg['kolba/naveska_s_rast_last']*(100-df_agg['koef_K'])
    df_agg['X2'] = df_agg['udelka'] * df_agg['zamer_2']/(df_agg['udelka']-1)/df_agg['kolba/naveska_s_rast_last']*(100-df_agg['koef_K'])
    df_agg['X3'] = df_agg['udelka'] * df_agg['zamer_3']/(df_agg['udelka']-1)/df_agg['kolba/naveska_s_rast_last']*(100-df_agg['koef_K'])

    return df_agg

def itog_raschet_gran(df_agg):
    df_agg['m_probi_bez_krupn'] = df_agg['kolba/naveska_last'] - df_agg[['gran_10_first', 'gran_5-10_first',
       'gran_5-2_first', 'gran_2-1_first']].sum(axis=1)

    df_agg['gran_10_%'] = df_agg['gran_10_first'] / df_agg['kolba/naveska_last'] * 100
    df_agg['gran_5-10_%'] = df_agg['gran_5-10_first'] / df_agg['kolba/naveska_last'] * 100
    df_agg['gran_5-2_%'] = df_agg['gran_5-2_first'] / df_agg['kolba/naveska_last'] * 100
    df_agg['gran_2-1_%'] = df_agg['gran_2-1_first'] / df_agg['kolba/naveska_last'] * 100

    df_agg['gran_1-0,5_%'] = df_agg['gran_1-0,5_first'] / df_agg['m_probi_bez_krupn'] * (100 - df_agg['koef_K'])
    df_agg['gran_0,5-0,25_%'] = df_agg['gran_0,5-0,25_first'] / df_agg['m_probi_bez_krupn'] * (100 - df_agg['koef_K'])
    df_agg['gran_0,25-0,10_%'] = df_agg['gran_0,25-0,10_first'] / df_agg['m_probi_bez_krupn'] * (100 - df_agg['koef_K'])

    df_agg['gran_0.05-0.01_%'] = df_agg['X1'] - df_agg['X2']
    df_agg['gran_0.01-0.002_%'] = df_agg['X2'] - df_agg['X3']
    df_agg['gran_0.002_%'] = df_agg['X3']

    df_agg['gran_0,10-0,05_%'] = 100 - df_agg[cols_kr_prozent].sum(axis=1) - df_agg[cols_melk_prozent].sum(axis=1)

    return df_agg

def rashet_gran(df_agg, df_tarirovk, udelka):
    df_agg = rashet_popravki_areometr(df_agg, df_tarirovk)

    df_agg = rashet_x1_x2_x3(df_agg, udelka)

    df_agg = itog_raschet_gran(df_agg)

    return df_agg

In [19]:
COlUMNS = {
    'Регистационный номер пробы': "lab_nomer",
    'Номер колбы_Масса возд. сух. навески, г': 'kolba/naveska_s_rast',
    'Номер колбы_Масса возд. сух. навески, г с учетом РО': "kolba/naveska",
    'Номер ареометра': 'areometr',
    '1 замер_Температура, °С': '1_zamer/temp',
    '2 замер_Температура, °С': '2_zamer/temp',
    '3 замер_Температура, °С': '3_zamer/temp',
    'Содержание растительных остатков': 'rast_ost',
    'Масса фракций, г_>10': 'gran_10',
    'Масса фракций, г_5-10': 'gran_5-10',
    'Масса фракций, г_5-2': 'gran_5-2',
    'Масса фракций, г_2-1': 'gran_2-1',
    'Масса фракций, г_1-0,5': 'gran_1-0,5',
    'Масса фракций, г_0,5-0,25': 'gran_0,5-0,25',
    'Масса фракций, г_0,25-0,10': 'gran_0,25-0,10',
}

COLS_GRAN = [
    'gran_10_first', 'gran_5-10_first',
    'gran_5-2_first', 'gran_2-1_first',
    'gran_1-0,5_first', 'gran_0,5-0,25_first',
    'gran_0,25-0,10_first'
]

agg_dict = {
    'kolba/naveska_s_rast': 'last',
    'kolba/naveska': 'last',
    'areometr': 'first',
    'rast_ost': 'first',
    'gran_10': 'first',
    'gran_5-10': 'first',
    'gran_5-2': 'first',
    'gran_2-1': 'first',
    'gran_1-0,5': 'first',
    'gran_0,5-0,25': 'first',
    'gran_0,25-0,10': 'first',
}

temp_agg = {
    '1_zamer/temp': ['first', 'last'],
    '2_zamer/temp': ['first', 'last'],
    '3_zamer/temp': ['first', 'last'],
}

cols_prozent_vse = [
    'lab_nomer',
    'gran_10_%',
    'gran_5-10_%',
    'gran_5-2_%',
    'gran_2-1_%',
    'gran_1-0,5_%',
    'gran_0,5-0,25_%',
    'gran_0,25-0,10_%',
    'gran_0,10-0,05_%',
    'gran_0.05-0.01_%',
    'gran_0.01-0.002_%',
    'gran_0.002_%'
]

cols_kr_prozent = [
    'gran_10_%',
    'gran_5-10_%',
    'gran_5-2_%',
    'gran_2-1_%',
    'gran_1-0,5_%',
    'gran_0,5-0,25_%',
    'gran_0,25-0,10_%'
]

cols_melk_prozent = [
    'gran_0.05-0.01_%',
    'gran_0.01-0.002_%',
    'gran_0.002_%'
]

In [22]:
PATH_NAMIV = "test_namiv2.xlsx"
PATH_TARIROVKI = "tarirovki.xlsx"
udelka = 2.7

df = zagr_file(PATH_NAMIV)
df_tarirovk = zagr_tarirovki(PATH_TARIROVKI)

df_agg = obrabotka_df_posle_zagr(df)

df_agg = rashet_gran(df_agg, df_tarirovk, udelka)


In [23]:
df_itog = df_agg[cols_prozent_vse].copy()
df_itog

,lab_nomer,gran_10_%,gran_5-10_%,gran_5-2_%,gran_2-1_%,"gran_1-0,5_%","gran_0,5-0,25_%","gran_0,25-0,10_%","gran_0,10-0,05_%",gran_0.05-0.01_%,gran_0.01-0.002_%,gran_0.002_%
1,26_09893,0.000000,0.000000,0.000000,0.099569,0.132850,0.332126,0.763890,12.248582,52.696941,26.348470,7.377572
2,26_09897,0.000000,0.000000,0.000000,0.000000,7.906054,6.086669,9.824677,12.611158,19.964585,23.116888,20.489969
3,26_09906,0.000000,0.000000,0.329381,0.032938,0.297193,0.561365,1.419924,6.433983,45.462608,36.579110,8.883498
4,26_09919,0.000000,0.000000,0.000000,0.000000,0.232404,0.199203,0.531208,4.650027,43.238809,31.110851,20.037497
5,26_09930,0.000000,1.146789,2.653997,3.604194,3.424013,2.040573,1.833057,11.037281,26.448801,26.957432,20.853862
...,...,...,...,...,...,...,...,...,...,...,...,...
60,26_10495,1.161248,2.720637,4.047777,3.715992,4.166861,2.934919,2.862452,11.273841,23.389004,25.422831,18.304438
61,26_10496,0.000000,0.000000,0.000000,0.099437,0.232180,0.132674,0.298517,8.192354,41.575388,25.260996,24.208454
62,26_10497,0.000000,6.490066,5.695364,6.456954,7.489683,5.722886,8.488307,12.012206,7.940756,14.888917,24.814862
63,26_10498,1.854305,2.582781,3.841060,3.741722,4.025995,2.937889,2.031133,16.140012,25.340767,21.793060,15.711276
